# 61. 대화코퍼스 컨텍스트 검색 (전체 음운현상)

30번(34-36) + 37번 사전검색 결과의 단어가 대화 코퍼스에서 실제로 어떤 맥락에서 사용되는지 확인

## 검색 대상 음운현상
- **34** ㄴ/ㄹ삽입
- **35** 유음화/비음화
- **36** 합성어경음화
- **37** 모음조화/충돌

## 단계
- **Stage A** (Colab): enriched CSV에서 검색 (샘플 3K or 전체 4.7GB)
- **Stage B** (Local): TextGrid/WAV 수집 스크립트 (collect_textgrid_wav.py)

## 입력
- 30번/37번 결과 CSV
- 대화 enriched: `01_nikl_dialogue_enriched_sample3k.csv` (샘플) 또는 전체

## 출력
- 전체 매칭 CSV: `search_results/dialogue_all_phenomena_*.csv`
  - `include`/`realized` 컬럼으로 수동 체크 후 TextGrid/WAV 수집

## 1. 환경 설정

In [55]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/DATA_2026'
except ImportError:
    PROJECT_ROOT = os.path.dirname(os.getcwd())
    print(f'Local mode: {PROJECT_ROOT}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [56]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re

# utils_phonology.py 로드
UTILS_PATH = f'{PROJECT_ROOT}/30_search_dictionary'
os.chdir(UTILS_PATH)
%run utils_phonology.py

# 경로 설정
SEARCH_RESULTS_30 = f'{PROJECT_ROOT}/30_search_dictionary/search_results'
SEARCH_RESULTS_37 = f'{PROJECT_ROOT}/37_vowel_harmony_collision/search_results'
RESULT_DIR = f'{PROJECT_ROOT}/60_search_dialogue_corpus/search_results'
os.makedirs(RESULT_DIR, exist_ok=True)

# 샘플/전체 전환
USE_SAMPLE = True  # True = 3K 샘플, False = 전체 4.7GB

if USE_SAMPLE:
    DIALOGUE_PATH = f'{PROJECT_ROOT}/00_raw_data/04_nikl_dialogue/02_csv/01_nikl_dialogue_enriched_sample3k.csv'
else:
    DIALOGUE_PATH = f'{PROJECT_ROOT}/00_raw_data/04_nikl_dialogue/02_csv/01_nikl_dialogue_enriched.csv'

print(f'USE_SAMPLE = {USE_SAMPLE}')
print(f'대화 CSV: {DIALOGUE_PATH}')

[OK] utils_phonology.py 로드 완료
   함수 34개
USE_SAMPLE = True
대화 CSV: /content/drive/MyDrive/DATA_2026/00_raw_data/04_nikl_dialogue/02_csv/01_nikl_dialogue_enriched_sample3k.csv


## 2. 데이터 로드

In [57]:
# 34-37번 결과에서 단어 목록 + 현상 태그 추출
PHENOMENON_MAP = {
    'n_l_insertion': '34_n_l_insertion',
    'nl_ln_nasalization': '35_nasalization',
    'fortis_compound': '36_fortis',
    'vowel_harmony_collision': '37_vowel',
    'vowel_collision': '37_vowel',
}

def classify_phenomenon(filename):
    for key, label in PHENOMENON_MAP.items():
        if key in filename:
            return label
    return 'unknown'

# 30번 결과
result_files_30 = list(Path(SEARCH_RESULTS_30).glob('*.csv'))
# 37번 결과 (사전검색 원본만)
result_files_37 = [f for f in Path(SEARCH_RESULTS_37).glob('*.csv')
                   if not f.name.startswith(('ls_freq_', 'seoul_', 'dialogue_'))]

# 단어 -> 현상 매핑
word_to_phenomena = {}
for f in result_files_30 + result_files_37:
    phenom = classify_phenomenon(f.stem)
    df_tmp = pd.read_csv(f, encoding='utf-8-sig', usecols=['word'])
    for w in df_tmp['word'].unique():
        if w not in word_to_phenomena:
            word_to_phenomena[w] = set()
        word_to_phenomena[w].add(phenom)

all_words = set(word_to_phenomena.keys())
print(f'전체 고유 단어: {len(all_words):,}개')
for phenom in sorted(set(p for ps in word_to_phenomena.values() for p in ps)):
    count = sum(1 for ps in word_to_phenomena.values() if phenom in ps)
    print(f'  {phenom}: {count:,}개')

전체 고유 단어: 204,656개
  34_n_l_insertion: 21,818개
  35_nasalization: 18,025개
  36_fortis: 118,354개
  37_vowel: 59,597개
  unknown: 17개


In [58]:
# 대화 코퍼스 로드
print(f'대화 코퍼스 로딩... ({DIALOGUE_PATH})')
df_dial = pd.read_csv(DIALOGUE_PATH, encoding='utf-8-sig', low_memory=False)
print(f'대화 발화: {len(df_dial):,}행')
print(f'컬럼: {list(df_dial.columns)}')

대화 코퍼스 로딩... (/content/drive/MyDrive/DATA_2026/00_raw_data/04_nikl_dialogue/02_csv/01_nikl_dialogue_enriched_sample3k.csv)
대화 발화: 2,999행
컬럼: ['file_id', 'year', 'category', 'doc_id', 'doc_title', 'doc_date', 'topic', 'speaker_id', 'speaker_age', 'speaker_sex', 'speaker_occupation', 'speaker_birthplace', 'speaker_principal_residence', 'speaker_current_residence', 'speaker_education', 'relation', 'utterance_id', 'form', 'original_form', 'start', 'end', 'note', 'pronunciation', 'form_roman', 'morphs', 'morphs_roman', 'morphs_v7_ids', 'morphs_v7_origins']


## 3. 단어 매칭 (form 한글 기반)

발화의 `form` 필드에서 어절을 분리하고, 사전검색 단어 목록과 매칭

In [59]:
def search_words_in_dialogue(df, word_to_phenomena):
    """
    대화 발화(form)에서 사전검색 단어 매칭

    form을 공백으로 split -> 각 어절이 word_to_phenomena에 있는지 확인
    매칭되면 해당 발화 정보 + 현상 태그를 반환
    """
    results = []

    for _, row in df.iterrows():
        form = str(row.get('form', ''))
        if not form or form == 'nan':
            continue

        eojeols = form.split()

        for eojeol in eojeols:
            # 정확히 일치하는 단어 검색
            if eojeol in word_to_phenomena:
                phenomena = ','.join(sorted(word_to_phenomena[eojeol]))
                results.append({
                    'utterance_id': row.get('utterance_id', ''),
                    'file_id': row.get('file_id', ''),
                    'start': row.get('start', ''),
                    'end': row.get('end', ''),
                    'matched_word': eojeol,
                    'phenomenon': phenomena,
                    'form': form,
                    'pronunciation': row.get('pronunciation', ''),
                    'speaker_id': row.get('speaker_id', ''),
                    'speaker_sex': row.get('speaker_sex', ''),
                    'speaker_age': row.get('speaker_age', ''),
                    'speaker_birthplace': row.get('speaker_birthplace', ''),
                    'include': '',
                    'realized': '',
                })

    return pd.DataFrame(results)

# 실행
print('대화 코퍼스에서 단어 검색 중...')
df_word_match = search_words_in_dialogue(df_dial, word_to_phenomena)
print(f'매칭 결과: {len(df_word_match):,}행')
print(f'매칭된 고유 단어: {df_word_match["matched_word"].nunique():,}개' if len(df_word_match) > 0 else '매칭 없음')

if len(df_word_match) > 0:
    print(f'\n현상별 매칭:')
    for phenom in sorted(set(p for ps in word_to_phenomena.values() for p in ps)):
        mask = df_word_match['phenomenon'].str.contains(phenom, na=False)
        print(f'  {phenom}: {mask.sum():,}행')
    print(f'\n상위 10건:')
    print(df_word_match[['matched_word', 'phenomenon', 'form']].head(10))

대화 코퍼스에서 단어 검색 중...
매칭 결과: 335행
매칭된 고유 단어: 213개

현상별 매칭:
  34_n_l_insertion: 55행
  35_nasalization: 29행
  36_fortis: 209행
  37_vowel: 50행
  unknown: 0행

상위 10건:
  matched_word        phenomenon                                          form
0           요새         36_fortis                                          요새 뭐
1         운영하다          37_vowel       저도 학원을 운영하다 보니까 코로나 때문에 많이 힘들어서 지금 뭐 투잡
2           투잡         36_fortis       저도 학원을 운영하다 보니까 코로나 때문에 많이 힘들어서 지금 뭐 투잡
3          숟가락         36_fortis                                이렇게 숟가락 얹기 식으로
4           병이  34_n_l_insertion  그냥 병이 있는 건 아닌데 항상 저희 집안에 이제 찬 바람을 직접 쐬면 안 되는
5           하다          37_vowel                              음 이렇게 대화를 하다 보니까
6          자외선         36_fortis                           훨씬 더 자외선 차단이 잘 될 거고
7           분이  34_n_l_insertion                               지인이나 가족 분이 있나요?
8           보기         36_fortis                                 가볍게 보기 좋은 영화고
9           삶이  34_n_l_insertion                 

In [60]:
# 화자별 통계
if len(df_word_match) > 0:
    print('=== 성별 분포 ===')
    print(df_word_match['speaker_sex'].value_counts())
    print(f'\n=== 연령 분포 ===')
    print(df_word_match['speaker_age'].value_counts())
    print(f'\n=== 출생지 분포 ===')
    print(df_word_match['speaker_birthplace'].value_counts().head(10))

=== 성별 분포 ===
speaker_sex
여성    181
남성    154
Name: count, dtype: int64

=== 연령 분포 ===
speaker_age
20대       103
30대        61
40대        55
10대        45
50대        45
60대 이상     22
60대         4
Name: count, dtype: int64

=== 출생지 분포 ===
speaker_birthplace
서울    71
경기    43
부산    31
대구    24
인천    20
대전    19
충남    19
경북    17
전북    16
경남    14
Name: count, dtype: int64


## 4. 결과 저장

In [61]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
suffix = '_sample' if USE_SAMPLE else ''

if len(df_word_match) > 0:
    out_path = f'{RESULT_DIR}/dialogue_all_phenomena{suffix}_{timestamp}.csv'
    df_word_match.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f'저장 완료: {out_path} ({len(df_word_match):,}행)')
else:
    print('매칭 결과 없음 - 저장 건너뜀')

print(f'\n다음 단계:')
print(f'1. CSV를 열어서 include 컬럼에 1 표시 (수집할 항목)')
print(f'2. realized 컬럼에 1/0 표시 (실현 여부)')
print(f'3. collect_textgrid_wav.py 실행하여 TextGrid/WAV 수집')

저장 완료: /content/drive/MyDrive/DATA_2026/60_search_dialogue_corpus/search_results/dialogue_all_phenomena_sample_20260313_052619.csv (335행)

다음 단계:
1. CSV를 열어서 include 컬럼에 1 표시 (수집할 항목)
2. realized 컬럼에 1/0 표시 (실현 여부)
3. collect_textgrid_wav.py 실행하여 TextGrid/WAV 수집


## 5. Stage B: TextGrid/WAV 수집 (로컬 실행)

Colab에서 여기까지 실행 후, 결과 CSV를 다운로드하여
Anaconda에서 `collect_textgrid_wav.py` 실행

```
python collect_textgrid_wav.py "결과CSV경로"
```

## 8. TODO: Stage B (로컬 TextGrid/WAV)

음향 분석이 필요한 경우:
- TextGrid: `D:\04_00_NIKL_DIALOGUE_MFA\06_textgrid_merged\` (40GB)
- WAV: `D:\04_00_NIKL_DIALOGUE_MFA\03_wav\` (431GB)
- 로컬 Jupyter에서 실행 필요 (D: 접근)